In [1]:
import pandas as pd
import numpy as np

# Configuration [cite: 494, 495]
STUDENT_ID = 5870 
df = pd.read_csv('most-popular-programming-languages-2004-2024.csv')
df['Month'] = pd.to_datetime(df['Month'])

# Assign Language [cite: 496]
languages = sorted([col for col in df.columns if col != 'Month'])
assigned_lang = languages[(STUDENT_ID % 1000) % len(languages)]

# Prep Data
lc_data = df[['Month', assigned_lang]].copy()
lc_data.rename(columns={assigned_lang: 'Popularity'}, inplace=True)

# Calculations [cite: 497, 498]
lc_data['Growth_Rate'] = lc_data['Popularity'].pct_change() * 100
lc_data['Moving_Avg'] = lc_data['Popularity'].rolling(window=6).mean()
lc_data['Moving_STD'] = lc_data['Popularity'].rolling(window=6).std()

# Statistical thresholds [cite: 499]
mean_growth = lc_data['Growth_Rate'].mean()
std_growth = lc_data['Growth_Rate'].std()

# Lifecycle Classification Rules [cite: 500, 501]
conditions = [
    (lc_data['Growth_Rate'] > mean_growth),                             # Growth
    (lc_data['Growth_Rate'] > 0) & (lc_data['Growth_Rate'] <= mean_growth), # Introduction
    (lc_data['Growth_Rate'].abs() <= 1.0),                             # Maturity
    (lc_data['Growth_Rate'] < 0) & (lc_data['Growth_Rate'] < -std_growth) # Decline
]
stages = ['Growth', 'Introduction', 'Maturity', 'Decline']
lc_data['Lifecycle_Phase'] = np.select(conditions, stages, default='Maturity')

# Summary Output [cite: 502, 503]
print(f"Analysis for: {assigned_lang}")
print(lc_data[['Month', 'Popularity', 'Growth_Rate', 'Lifecycle_Phase']].tail())
print("\nPhase Distribution:")
print(lc_data['Lifecycle_Phase'].value_counts(normalize=True) * 100)

Analysis for: C# Worldwide(%)
         Month  Popularity  Growth_Rate Lifecycle_Phase
244 2024-05-01          32     0.000000          Growth
245 2024-06-01          30    -6.250000        Maturity
246 2024-07-01          29    -3.333333        Maturity
247 2024-08-01          27    -6.896552         Decline
248 2024-09-01          27     0.000000          Growth

Phase Distribution:
Lifecycle_Phase
Growth      54.216867
Maturity    29.317269
Decline     16.465863
Name: proportion, dtype: float64
